In [1]:
import nlpUtility as nlpUtil
import torch.nn as nn
import torch

In [2]:
review_max_length = 200
vocab_size = 10000
embed_dim = 128

In [3]:
train_loader, test_loader, vocab, vocab_size = nlpUtil.data_pipline('NLPdata/IMDB Dataset.csv', review_max_length, vocab_size)

In [14]:
for reviews, labels in test_loader:
    print("Test batch shapes:")
    print("reviews:", reviews.shape)
    print("labels:", labels.shape)
    break

Test batch shapes:
reviews: torch.Size([32, 200])
labels: torch.Size([32])


In [4]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        
        self.fc = nn.Linear(hidden_dim, 1)  

    def forward(self, x):
        x = self.embedding(x)  # (batch, seq_len, embed_dim)
        
        lstm_out, (h_n, c_n) = self.lstm(x)
        
        # Take last hidden state
        out = h_n[-1]  # (batch, hidden_dim)
        
        out = self.fc(out)  # (batch, 1)
        
        return out

In [7]:
model = SentimentLSTM(vocab_size, embed_dim, hidden_dim=64)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [8]:
from tqdm.auto import tqdm

for epoch in range(3):  # 3 epochs for example
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
    for x, y in pbar:
        optimizer.zero_grad()
        output = model(x)             # forward
        y = y.float().unsqueeze(1)    # Ensure labels are float for BCEWithLogitsLoss
        loss = criterion(output, y)   # compute loss
        loss.backward()               # backprop
        optimizer.step()              # update weights

        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.6888


Epoch 2, Loss: 0.6648


Epoch 3, Loss: 0.6416


In [11]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for review, label in test_loader:
        output = model(review)
        probs = torch.sigmoid(output)
        preds = (probs > 0.5).long()    
        label = label.unsqueeze(1)  # Ensure label shape matches preds
        correct += (preds == label).sum().item()
        total += label.size(0)

In [12]:
print("Test accuracy:", (correct / total) * 100, "%")

Test accuracy: 65.34 %
